[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/06_multihead_attention.ipynb)

# 🔴 Hard: Multi-Head Attention

Implement **Multi-Head Attention** from scratch — the core building block of the Transformer.

$$\text{MultiHead}(Q, K, V) = \text{Concat}(\text{head}_1, \dots, \text{head}_h) W^O$$
$$\text{head}_i = \text{Attention}(Q W_i^Q,\; K W_i^K,\; V W_i^V)$$

### Signature
```python
class MultiHeadAttention:
    def __init__(self, d_model: int, num_heads: int): ...
    def forward(self, Q, K, V) -> torch.Tensor: ...
```

### Requirements
- Use `nn.Linear(d_model, d_model)` for `self.W_q`, `self.W_k`, `self.W_v`, `self.W_o`
- `d_k = d_model // num_heads` per head
- `forward(Q, K, V)`: Q is `(B, seq_q, d_model)`, K/V are `(B, seq_k, d_model)`
- Must support **cross-attention** (`seq_q != seq_k`)
- Do **NOT** use `torch.nn.MultiheadAttention`
- You **may** use `torch.softmax` and `torch.matmul`

### Steps
1. Project: `q = self.W_q(Q)`, `k = self.W_k(K)`, `v = self.W_v(V)`
2. Reshape to `(B, num_heads, seq, d_k)`
3. Scaled dot-product attention per head
4. Concat heads → `(B, seq_q, d_model)`
5. Output projection: `self.W_o(concat)`

### Multi-Head Attention: Tensor Shape Cheat Sheet

**Notation Key:**
* `B` = Batch size
* `seq_q` = Sequence length of Query
* `seq_k` = Sequence length of Key and Value (may differ from `seq_q` in cross-attention)
* `d_model` = Total embedding dimension
* `H` = Number of heads (`num_heads`)
* `d_k` = Dimension per head (`d_model // num_heads`)

---

#### 1. Initial Inputs & Weights
* **`Q`**: `(B, seq_q, d_model)`
* **`K`, `V`**: `(B, seq_k, d_model)`
* **Linear Weights (`W_q`, `W_k`, `W_v`, `W_o`)**: `(d_model, d_model)`

#### 2. After Linear Projection
* **`QW`**: `(B, seq_q, d_model)`
* **`KW`, `VW`**: `(B, seq_k, d_model)`

#### 3. Reshape & Transpose (Splitting into Heads)
* *Intermediate Step (`.view`)*: `(B, seq, H, d_k)`
* *Final Step (`.transpose(1, 2)`)*:
  * **`QW`**: `(B, H, seq_q, d_k)`
  * **`KW`, `VW`**: `(B, H, seq_k, d_k)`

#### 4. Scaled Dot-Product Attention (Per Head)
* **`KW.transpose(-2, -1)`**: `(B, H, d_k, seq_k)`
* **`QK` (Raw Attention Scores)**: `(B, H, seq_q, seq_k)`
  * *Derivation:* `(..., seq_q, d_k) @ (..., d_k, seq_k)`
* **`att_weights` (Probabilities)**: `(B, H, seq_q, seq_k)`
* **`output` (Weighted Values)**: `(B, H, seq_q, d_k)`
  * *Derivation:* `(..., seq_q, seq_k) @ (..., seq_k, d_k)`

#### 5. Concatenation & Output Projection
* **`output.transpose(1, 2)`** (Prep for concat): `(B, seq_q, H, d_k)`
* **`.reshape(..., -1, d_model)`** (Flatten/Concat): `(B, seq_q, d_model)`
* **`final_output`** (After `self.W_o`): `(B, seq_q, d_model)`

In [2]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 2.8 MB/s eta 0:00:00


In [3]:
import torch
import torch.nn as nn
import math

In [11]:
# ✏️ YOUR IMPLEMENTATION HERE

class MultiHeadAttention:
    def __init__(self, d_model: int, num_heads: int):
        # pass  # Initialize W_q, W_k, W_v, W_o
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        self.W_q = torch.nn.Linear(d_model, d_model)
        self.W_k = torch.nn.Linear(d_model, d_model)
        self.W_v = torch.nn.Linear(d_model, d_model)
        self.W_o = torch.nn.Linear(d_model, d_model)



    def forward(self, Q, K, V):
        # pass  # Implement multi-head attention
        QW = self.W_q(Q)
        KW = self.W_k(K)
        VW = self.W_v(V)
        QW = QW.view(QW.shape[0], QW.shape[1],self.num_heads, self.d_k).transpose(1, 2)
        KW = KW.view(KW.shape[0], KW.shape[1], self.num_heads, self.d_k).transpose(1, 2)
        VW = VW.view(VW.shape[0], VW.shape[1], self.num_heads, self.d_k).transpose(1, 2)

        QK = torch.matmul(QW, KW.transpose(-2, -1))
        score = QK / math.sqrt(self.d_k)
        att_weights = torch.softmax(score, dim = -1)

        output = att_weights.matmul(VW)
        final_output = self.W_o((output).transpose(1, 2).reshape(output.shape[0], -1, self.d_model))
        return final_output


In [12]:
# 🧪 Debug
torch.manual_seed(0)
mha = MultiHeadAttention(d_model=32, num_heads=4)
print("W_q type:", type(mha.W_q))          # should be nn.Linear
print("W_q.weight shape:", mha.W_q.weight.shape)  # (32, 32)

x = torch.randn(2, 6, 32)
out = mha.forward(x, x, x)
print("Output shape:", out.shape)          # (2, 6, 32)

# Cross-attention
Q = torch.randn(1, 3, 32)
K = torch.randn(1, 7, 32)
V = torch.randn(1, 7, 32)
out2 = mha.forward(Q, K, V)
print("Cross-attn shape:", out2.shape)     # (1, 3, 32)

W_q type: <class 'torch.nn.modules.linear.Linear'>
W_q.weight shape: torch.Size([32, 32])
Output shape: torch.Size([2, 6, 32])
Cross-attn shape: torch.Size([1, 3, 32])


In [13]:
# ✅ SUBMIT
from torch_judge import check
check("mha")


🧪 Testing: Multi-Head Attention (Hard)
──────────────────────────────────────────────────
  ✅ [1/6] Output shape (3.5ms)
  ✅ [2/6] Uses nn.Linear with correct shapes (0.9ms)
  ✅ [3/6] Numerical correctness vs reference (30.1ms)
  ✅ [4/6] Gradient flow (32.7ms)
  ✅ [5/6] Cross-attention (seq_q != seq_k) (1.3ms)
  ✅ [6/6] Different heads give different outputs (3.0ms)
──────────────────────────────────────────────────
  🎉 All 6 tests passed! (71.6ms total)
  Progress saved. Run status() to see your dashboard.

